# 🚀 FRACTAL GPU Runtime - Colab Export

Export PyTorch transformer models to XJSON format for local testing with WebGPU runtime.

**Workflow:**
1. Train/load model in Colab (this notebook)
2. Export to XJSON format
3. Download and test locally with WebGPU runtime
4. Fine-tune in Colab
5. Re-export and test

**Compatible with:**
- GPT-2, GPT-Neo, GPT-J
- Custom transformer architectures
- HuggingFace models

In [ ]:
# Install dependencies
!pip install torch transformers accelerate -q

## 1. Define Transformer Model

In [ ]:
import torch
import torch.nn as nn
import json
import numpy as np
from typing import Dict, Any

class SimpleTransformer(nn.Module):
    """Minimal transformer for testing"""
    def __init__(self, vocab_size=1000, n_embd=128, n_head=4, n_layer=2, max_seq_len=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.n_head = n_head
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(max_seq_len, n_embd)
        
        # Transformer layers
        self.layers = nn.ModuleList([
            TransformerBlock(n_embd, n_head) for _ in range(n_layer)
        ])
        
        # Final layer norm and projection
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
    def forward(self, input_ids):
        B, T = input_ids.shape
        
        # Embeddings
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # Transformer blocks
        for layer in self.layers:
            x = layer(x)
        
        # Output
        x = self.final_norm(x)
        logits = self.lm_head(x)
        
        return logits

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd)
        )
        
    def forward(self, x):
        # Self-attention with residual
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        # FFN with residual
        x = x + self.ffn(self.ln2(x))
        return x

print("✅ Model classes defined")

## 2. Create or Load Model

In [ ]:
# Option A: Create new model
model = SimpleTransformer(
    vocab_size=1000,
    n_embd=128,
    n_head=4,
    n_layer=2,
    max_seq_len=256
)

# Option B: Load pre-trained weights (uncomment if you have a checkpoint)
# model.load_state_dict(torch.load('model_checkpoint.pt'))

model.eval()
print(f"✅ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"   Config: {model.n_layer} layers, {model.n_embd}d, {model.n_head} heads")

## 3. Export to XJSON Format

In [ ]:
def export_to_xjson(model: nn.Module, output_path: str = 'model.xjson') -> Dict[str, Any]:
    """
    Export PyTorch model to XJSON format for FRACTAL GPU Runtime
    """
    print("🔄 Exporting model to XJSON...")
    
    # Extract configuration
    config = {
        'vocab_size': model.vocab_size,
        'n_embd': model.n_embd,
        'n_head': model.n_head,
        'n_layer': model.n_layer,
        'max_seq_len': model.max_seq_len,
        'activation': 'gelu'
    }
    
    # Build layer structure
    layers = [
        {
            'type': 'embedding',
            'name': 'token_embedding',
            'params': {'vocab_size': config['vocab_size'], 'embedding_dim': config['n_embd']}
        },
        {
            'type': 'embedding',
            'name': 'position_embedding',
            'params': {'max_positions': config['max_seq_len'], 'embedding_dim': config['n_embd']}
        }
    ]
    
    # Add transformer blocks
    for i in range(config['n_layer']):
        layers.append({
            'type': 'transformer_block',
            'name': f'layer_{i}',
            'params': {
                'n_embd': config['n_embd'],
                'n_head': config['n_head'],
                'activation': 'gelu'
            }
        })
    
    # Add final layers
    layers.extend([
        {'type': 'layer_norm', 'name': 'final_norm', 'params': {'n_embd': config['n_embd']}},
        {'type': 'linear', 'name': 'lm_head', 'params': {'in_features': config['n_embd'], 'out_features': config['vocab_size']}}
    ])
    
    # Extract weights
    weights = {}
    state_dict = model.state_dict()
    
    print(f"   Extracting {len(state_dict)} weight tensors...")
    
    for name, param in state_dict.items():
        # Convert to numpy and then to list for JSON serialization
        weight_data = param.cpu().detach().numpy()
        
        # For large models, optionally quantize to reduce size
        # weight_data = weight_data.astype(np.float16)  # Half precision
        
        weights[name] = {
            'data': weight_data.tolist(),
            'shape': list(weight_data.shape),
            'dtype': 'float32'
        }
        
        print(f"   ✓ {name}: shape={weight_data.shape}")
    
    # Build XJSON structure
    xjson = {
        'fractal_runtime': '1.0',
        'model_type': 'transformer',
        'config': config,
        'architecture': 'transformer',
        'layers': layers,
        'weights': weights,
        'metadata': {
            'name': 'exported-transformer',
            'version': '0.1.0',
            'framework': 'pytorch',
            'exported_from': 'colab',
            'total_params': sum(p.numel() for p in model.parameters()),
            'compatible_with': [
                '@fractal/gpu-runtime',
                '@xjson/xjson-server',
                '@xjson/klh-orchestrator'
            ]
        }
    }
    
    # Save to file
    print(f"\n💾 Saving to {output_path}...")
    with open(output_path, 'w') as f:
        json.dump(xjson, f, indent=2)
    
    # File size
    import os
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ Export complete! File size: {size_mb:.2f} MB")
    
    return xjson

# Export the model
xjson_model = export_to_xjson(model, 'fractal_model.xjson')

## 4. Verify Export

In [ ]:
# Load and verify
with open('fractal_model.xjson', 'r') as f:
    loaded = json.load(f)

print("✅ XJSON Model Verification")
print(f"   Model type: {loaded['model_type']}")
print(f"   Architecture: {loaded['architecture']}")
print(f"   Layers: {loaded['config']['n_layer']}")
print(f"   Hidden size: {loaded['config']['n_embd']}")
print(f"   Attention heads: {loaded['config']['n_head']}")
print(f"   Vocab size: {loaded['config']['vocab_size']}")
print(f"   Total params: {loaded['metadata']['total_params']:,}")
print(f"   Weight tensors: {len(loaded['weights'])}")

print("\n📋 Layer Structure:")
for i, layer in enumerate(loaded['layers']):
    print(f"   {i}. {layer['type']} ({layer['name']})")

## 5. Download Model

Download `fractal_model.xjson` to your local machine and test with WebGPU runtime:

```bash
# On your local machine
cd FRACTAL-GPU-RUNTIME
node examples/load-xjson.js fractal_model.xjson
```

In [ ]:
# Download file
from google.colab import files
files.download('fractal_model.xjson')
print("📥 Download started! Check your browser downloads.")

## 6. Optional: Quantization for Smaller Files

In [ ]:
def export_quantized(model: nn.Module, output_path: str = 'model_int8.xjson', bits: int = 8):
    """
    Export with INT8 or INT4 quantization for smaller file size
    """
    print(f"🔄 Exporting with {bits}-bit quantization...")
    
    xjson = export_to_xjson(model, output_path + '.tmp')
    
    # Quantize weights
    quantized_weights = {}
    total_size_reduction = 0
    
    for name, weight_info in xjson['weights'].items():
        data = np.array(weight_info['data'])
        original_size = data.nbytes
        
        if bits == 8:
            # INT8 quantization
            min_val, max_val = data.min(), data.max()
            scale = (max_val - min_val) / 255.0
            zero_point = -min_val / scale
            
            quantized = np.clip(np.round(data / scale + zero_point), 0, 255).astype(np.uint8)
            
            quantized_weights[name] = {
                'data': quantized.tolist(),
                'shape': weight_info['shape'],
                'dtype': 'int8',
                'scale': float(scale),
                'zero_point': float(zero_point)
            }
        
        new_size = quantized.nbytes
        total_size_reduction += (original_size - new_size)
        print(f"   ✓ {name}: {original_size/1024:.1f}KB → {new_size/1024:.1f}KB")
    
    xjson['weights'] = quantized_weights
    xjson['metadata']['quantization'] = f'int{bits}'
    
    # Save
    with open(output_path, 'w') as f:
        json.dump(xjson, f, indent=2)
    
    import os
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\n✅ Quantized export complete!")
    print(f"   File size: {size_mb:.2f} MB")
    print(f"   Size reduction: {total_size_reduction/(1024*1024):.2f} MB")
    
    return xjson

# Export quantized version (optional)
# quantized_model = export_quantized(model, 'fractal_model_int8.xjson', bits=8)

## 7. Test Inference in Colab

In [ ]:
# Quick inference test
model.eval()

with torch.no_grad():
    input_ids = torch.randint(0, model.vocab_size, (1, 10))  # Batch size 1, seq len 10
    logits = model(input_ids)
    
    print("✅ Inference Test")
    print(f"   Input shape: {input_ids.shape}")
    print(f"   Output shape: {logits.shape}")
    print(f"   Expected: [1, 10, {model.vocab_size}]")
    print(f"\n   Sample logits: {logits[0, 0, :5].tolist()}")

## 📝 Next Steps

1. **Download** the exported `fractal_model.xjson`
2. **Test locally** with WebGPU runtime:
   ```bash
   node examples/load-xjson.js fractal_model.xjson
   ```
3. **Fine-tune** in Colab (see fine-tuning notebook)
4. **Re-export** and test updated weights
5. **Deploy** with @xjson/klh-orchestrator

## 🔗 Resources

- [FRACTAL GPU Runtime Repo](https://github.com/cannaseedus-bot/FRACTAL-GPU-RUNTIME)
- [@xjson/xjson-server](https://www.npmjs.com/package/@xjson/xjson-server)
- [@xjson/klh-orchestrator](https://www.npmjs.com/package/@xjson/klh-orchestrator)